# Potato Disease Analysis with Grad-CAM

This notebook demonstrates three key features:
1. **Grad-CAM Visualization** - See which regions the model focuses on
2. **Disease Region Masking** - Isolate affected areas from healthy tissue
3. **Disease Severity Quantification** - Measure how severe the disease is

## 1. Setup and Imports

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import our custom utilities
from gradcam_utils import GradCAMAnalyzer, load_and_analyze

print(f"TensorFlow version: {tf.__version__}")
print("Setup complete!")

## 2. Load Model and Data

In [ ]:
# Load a trained model
# Options: saved_models/1 (CNN), saved_models/2 (ResNet), saved_models/3 (MobileNetV2)
MODEL_PATH = Path("../saved_models/1")  # CNN Baseline

CLASS_NAMES = ["Early Blight", "Late Blight", "Healthy"]

model = tf.keras.models.load_model(str(MODEL_PATH))
print(f"Loaded model from {MODEL_PATH}")
print(f"Input shape: {model.input_shape}")
print(f"Output classes: {CLASS_NAMES}")

In [ ]:
# Create the analyzer
analyzer = GradCAMAnalyzer(
    model=model,
    class_names=CLASS_NAMES,
    image_size=256,
    mask_threshold=0.5,
    heatmap_threshold=0.4
)
print("GradCAMAnalyzer initialized!")

In [ ]:
# Helper function to load images
def load_image(path):
    """Load and preprocess an image."""
    return np.array(
        Image.open(path).convert('RGB').resize((256, 256), Image.Resampling.LANCZOS)
    )

# Load sample images (update paths to your actual images)
# You can use images from the PlantVillage dataset or your own photos
sample_images = {}

# Try to find sample images
dataset_path = Path("../PlantVillage")
if dataset_path.exists():
    # Find one example from each class
    for class_name in ["Potato___Early_blight", "Potato___Late_blight", "Potato___healthy"]:
        class_path = dataset_path / class_name
        if class_path.exists():
            image_files = list(class_path.glob("*.jpg"))[:1]
            if image_files:
                clean_name = class_name.replace("Potato___", "").replace("_", " ")
                sample_images[clean_name] = load_image(image_files[0])
                print(f"Loaded {clean_name}: {image_files[0].name}")
else:
    print("Dataset not found. Please provide your own images.")
    print("Example: sample_images['Early Blight'] = load_image('path/to/image.jpg')")

print(f"\nLoaded {len(sample_images)} sample images")

## 3. Grad-CAM Visualization

Grad-CAM (Gradient-weighted Class Activation Mapping) shows which regions of the image
the model focuses on when making its prediction. Hot regions (red) indicate areas
that strongly influence the prediction.

In [ ]:
# Analyze a single image
if sample_images:
    # Get the first available image
    image_name, image = next(iter(sample_images.items()))
    
    # Run analysis
    result = analyzer.analyze(image, include_mask=False, include_severity=False)
    
    # Visualize
    analyzer.visualize(result, show_severity=False)
    
    print(f"\nPrediction: {result.prediction} ({result.confidence:.1%})")
    print(f"Class probabilities:")
    for cls, prob in result.probabilities.items():
        print(f"  {cls}: {prob:.1%}")

### Understanding the Visualization

- **Original Image**: The input potato leaf image
- **Grad-CAM Overlay**: Heatmap blended with original image. Red/warm areas are what the model "looks at"
- **Raw Heatmap**: The heatmap alone, showing activation intensity

## 4. Disease Region Masking

The disease mask isolates the affected regions from healthy tissue.
This is useful for:
- Quantifying the extent of disease
- Visual inspection of affected areas
- Precise disease localization

In [ ]:
if sample_images:
    # Get an image with disease (not healthy)
    disease_image = None
    disease_name = None
    for name, img in sample_images.items():
        if "healthy" not in name.lower():
            disease_image = img
            disease_name = name
            break
    
    if disease_image is None:
        # Use first available image
        disease_name, disease_image = next(iter(sample_images.items()))
    
    print(f"Analyzing: {disease_name}")
    
    # Full analysis with masking
    result = analyzer.analyze(disease_image)
    
    # Visualize all components
    analyzer.visualize(result, show_severity=True)
    
    # Show mask creation process
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    axes[0].imshow(result.image)
    axes[0].set_title("Original")
    axes[0].axis('off')
    
    heatmap_resized = tf.image.resize(
        result.heatmap_raw[..., np.newaxis], (256, 256)
    ).numpy().squeeze()
    axes[1].imshow(heatmap_resized, cmap='jet', vmin=0, vmax=1)
    axes[1].set_title("Heatmap")
    axes[1].axis('off')
    
    axes[2].imshow(result.mask, cmap='gray')
    axes[2].set_title("Binary Mask")
    axes[2].axis('off')
    
    axes[3].imshow(result.masked_image)
    axes[3].set_title("Disease Regions Only")
    axes[3].axis('off')
    
    plt.suptitle(f"Disease Region Masking: {disease_name}", fontsize=14)
    plt.tight_layout()
    plt.show()

### Different Masking Methods

The analyzer supports three masking methods:
- **threshold**: Simple thresholding (default)
- **otsu**: Automatic threshold using Otsu's method
- **adaptive**: Adaptive thresholding based on local mean

In [ ]:
# Compare different masking methods
if sample_images and disease_image is not None:
    # Compute heatmap once
    heatmap, _ = analyzer.compute_gradcam(disease_image)
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    methods = ["threshold", "otsu", "adaptive"]
    
    axes[0].imshow(disease_image)
    axes[0].set_title("Original")
    axes[0].axis('off')
    
    for i, method in enumerate(methods):
        mask = analyzer.create_disease_mask(heatmap, method=method)
        masked_img = analyzer.create_masked_image(disease_image, mask)
        axes[i + 1].imshow(masked_img)
        axes[i + 1].set_title(f"Method: {method}")
        axes[i + 1].axis('off')
    
    plt.suptitle("Comparison of Masking Methods", fontsize=14)
    plt.tight_layout()
    plt.show()

## 5. Disease Severity Quantification

The severity analysis provides:
- **Affected Area**: Percentage of the leaf affected
- **Mean Intensity**: Average heatmap intensity in affected regions
- **Severity Level**: Classification (healthy, mild, moderate, severe)
- **Intensity-Weighted Severity**: Combined metric of area and intensity

In [ ]:
# Analyze severity for all sample images
if sample_images:
    print("=" * 70)
    print("DISEASE SEVERITY ANALYSIS")
    print("=" * 70)
    
    severity_results = []
    
    for name, image in sample_images.items():
        result = analyzer.analyze(image)
        severity_results.append((name, result))
        
        print(f"\n{name}:")
        print(f"  Prediction: {result.prediction} ({result.confidence:.1%})")
        print(f"  Affected Area: {result.severity['affected_percentage']:.1f}%")
        print(f"  Mean Intensity: {result.severity['mean_intensity']:.3f}")
        print(f"  Severity Level: {result.severity['severity_level'].upper()}")
        print(f"  Intensity-Weighted: {result.severity['intensity_weighted_severity']:.2f}")
    
    print("\n" + "=" * 70)

In [ ]:
# Visual comparison of severity across samples
if len(severity_results) >= 2:
    analyzer.visualize_comparison(
        [r for _, r in severity_results],
        titles=[n for n, _ in severity_results]
    )

## 6. Model Comparison

Compare how different models visualize the same image.

In [ ]:
# Compare different models on the same image
model_configs = [
    ("../saved_models/1", "CNN Baseline"),
    ("../saved_models/2", "Transfer Learning"),
    ("../saved_models/3", "MobileNetV2"),
]

# Use first available sample image
if sample_images:
    test_name, test_image = next(iter(sample_images.items()))
    
    print(f"Comparing models on: {test_name}\n")
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    for col, (path, name) in enumerate(model_configs):
        try:
            m = tf.keras.models.load_model(path)
            a = GradCAMAnalyzer(m, CLASS_NAMES)
            result = a.analyze(test_image)
            
            axes[0, col].imshow(result.heatmap_overlay)
            axes[0, col].set_title(f"{name}\n{result.prediction} ({result.confidence:.1%})")
            axes[0, col].axis('off')
            
            axes[1, col].imshow(result.masked_image)
            affected = result.severity.get('affected_percentage', 0)
            severity = result.severity.get('severity_level', 'N/A')
            axes[1, col].set_title(f"Severity: {severity}\nAffected: {affected:.1f}%")
            axes[1, col].axis('off')
            
            print(f"{name}: {result.prediction} ({result.confidence:.1%}), Severity: {severity}")
            
        except Exception as e:
            print(f"Failed to load {name}: {e}")
            axes[0, col].text(0.5, 0.5, f"Error:\n{e}", ha='center', va='center')
            axes[0, col].axis('off')
            axes[1, col].axis('off')
    
    plt.suptitle(f"Model Comparison: {test_name}", fontsize=14)
    plt.tight_layout()
    plt.show()

## 7. Batch Analysis

Analyze multiple images and aggregate severity statistics.

In [ ]:
# Batch analysis function
def batch_analyze(image_dict, analyzer):
    """Analyze multiple images and return aggregated statistics."""
    results = []
    for name, image in image_dict.items():
        result = analyzer.analyze(image)
        results.append({
            'name': name,
            'prediction': result.prediction,
            'confidence': result.confidence,
            'affected_percentage': result.severity['affected_percentage'],
            'severity_level': result.severity['severity_level'],
            'mean_intensity': result.severity['mean_intensity'],
        })
    return results

# Run batch analysis
if sample_images:
    batch_results = batch_analyze(sample_images, analyzer)
    
    # Print summary table
    print("\n" + "=" * 80)
    print(f"{'Image':<30} {'Prediction':<15} {'Confidence':<12} {'Affected':<10} {'Severity':<12}")
    print("=" * 80)
    
    for r in batch_results:
        print(
            f"{r['name']:<30} "
            f"{r['prediction']:<15} "
            f"{r['confidence']:<12.1%} "
            f"{r['affected_percentage']:<10.1f} "
            f"{r['severity_level']:<12}"
        )
    
    print("=" * 80)

## 8. Custom Image Analysis

Use this section to analyze your own images.

In [ ]:
# Analyze a custom image
# Uncomment and update the path below

# custom_image_path = "path/to/your/potato_leaf.jpg"
# custom_image = load_image(custom_image_path)
# result = analyzer.analyze(custom_image)
# analyzer.visualize(result)

## Summary

### Key Features Implemented:

1. **Grad-CAM Visualization**
   - Shows which image regions influence predictions
   - Works with all model architectures
   - Includes raw heatmap and overlay options

2. **Disease Region Masking**
   - Binary mask isolates affected tissue
   - Three methods: threshold, Otsu, adaptive
   - Morphological operations clean up noise

3. **Disease Severity Quantification**
   - Affected area percentage
   - Mean/max intensity metrics
   - Severity levels: healthy, mild, moderate, severe
   - Intensity-weighted combined score

### Usage:

```python
from gradcam_utils import GradCAMAnalyzer

analyzer = GradCAMAnalyzer(model, class_names=[...])
result = analyzer.analyze(image)
analyzer.visualize(result)
print(result.severity)
```